# 02 · ¿Qué método usar para este hueco?
Primero se aprende y evalúa la regla ocultando concentraciones conocidas. La aplicación a ausencias reales es una segunda etapa. Por defecto se leen los resultados agregados publicados; las dos opciones permiten reproducirlos desde los Excel.

In [ ]:
import json
import pandas as pd
from protocolo.datos import PRIV, PUB

RECALCULAR_VALIDACION = False
RECALCULAR_APLICACION = False
PROCESOS = 4

## 1. Validar con mediciones que sí conocemos
Cinco bloques externos separan perturbaciones; tres internos eligen la opción por objetivo y perfil. El código aprende los predictores, el escalado y las reglas dentro del entrenamiento. Se prueban una ausencia individual, patrones conjuntos observados y exclusión por serie. Para recalcular, ejecutar primero el notebook 01.

In [ ]:
if RECALCULAR_VALIDACION:
    from protocolo.validar import validar
    from protocolo.informe import metricas_validacion
    validar(procesos=PROCESOS)
    metricas_validacion()

errores = pd.read_csv(PUB / 'metricas.csv')
iguales = errores.query("prueba == 'perturbacion' and tipo == 'individual' and alcance == 'mismas_celdas'")
display(iguales[['metodo', 'celdas_distintas', 'grupos', 'MAE_mM', 'MAE_macro_mM', 'RMSE_mM', 'sesgo_mM']])
iguales.set_index('metodo')[['MAE_mM', 'MAE_macro_mM']].rename(columns={'MAE_mM':'Micro', 'MAE_macro_mM':'Macro'}).plot.barh(xlabel='MAE (mM)', title='Mismas mediciones ocultas');

**¿Aporta MMC2?** Se compara el mismo método y panel químico en las mismas mediciones, con y sin las ocho condiciones auxiliares. Un delta negativo favorece añadir MMC2. Revisar también cuántos grupos mejoran.

In [ ]:
pares = pd.read_csv(PUB / 'MMC2_pareado.csv')
casos = ['fum', 'cit', 'ade', 'atp', 'pep', 's7p']
display(pares[pares.metodo.eq('Iterativo') & pares.objetivo.isin(casos)][['objetivo', 'celdas', 'grupos', 'MAE_control_mM', 'MAE_mM', 'delta_MAE_mM', 'grupos_con_menor_MAE']])

## 2. Aplicar la regla congelada a los huecos reales
Activar la segunda opción solo tras validar las mismas entradas. Nunca se alimenta un método con la matriz ya imputada. Los códigos de ausencia originales se conservan. Las medianas de respaldo se identifican por separado.

In [ ]:
if RECALCULAR_APLICACION:
    from protocolo.aplicar import aplicar
    from protocolo.intervalos import intervalos
    from protocolo.informe import informe
    aplicar()
    intervalos(procesos=PROCESOS)
    informe()

resultado = json.loads((PUB / 'ejecucion.json').read_text())
display(pd.Series(resultado['por_estado'], name='huecos').to_frame())

**¿Qué significa un intervalo?** Solo el iterativo lo produce. Los percentiles 2,5 y 97,5 delimitan el 95 % central de los sorteos del modelo. En la prueba podemos contar qué fracción de los intervalos contiene la verdad. En los huecos reales esa verdad se desconoce.

In [ ]:
intervalos_publicados = pd.read_csv(PUB / 'intervalos.csv')
display(intervalos_publicados[['prueba', 'tipo', 'candidatas', 'con_prediccion', 'con_intervalo', 'dentro', 'cobertura_entre_intervalos', 'ancho_mediano_mM']])

**Antes y después.** El resumen «disponible» usa lo que aporta cada método. «huecos_comunes» permite comparar usando solo las mismas casillas. El CV se expresa en porcentaje. Para ver un cultivo concreto abre el archivo de auditoría generado.

In [ ]:
estadisticas = pd.read_csv(PUB / 'descriptivos.csv')
display(estadisticas.query("objetivo == 'fum' and alcance == 'huecos_comunes'")[[ 'metodo', 'observados', 'imputados', 'pendientes', 'media_mM', 'mediana_mM', 'SD_mM', 'IQR_mM', 'CV_pct']])
if (PRIV / 'auditoria_huecos.csv').exists():
    detalle = pd.read_csv(PRIV / 'auditoria_huecos.csv')
    display(detalle.loc[detalle.objetivo.eq('fum'), ['cultivo', 'source_cell', 'estado', 'metodo', 'pred', 'motivo', 'n_celdas_comparables']].head())